# Test 07 Reconciliation
Checks that every Bronze row is accounted for in Silver or quarantine, and that Silver keys are unique.

In [0]:
from datetime import datetime

run_ts = datetime.now()
results = []

def add(table, name, n):   # n = failing rows, 0 = pass
    results.append((table, name, int(n), "PASS" if n == 0 else "FAIL", run_ts))

def one(sql):
    return spark.sql(sql).collect()[0][0]

# 1. counts: Bronze = Silver + quarantine
for name, b, s, q in [("trades",  "bronze.trades",  "silver.trades",  "silver.trades_quarantine"),
                      ("prices",  "bronze.prices",  "silver.prices",  "silver.prices_quarantine")]:
    bc, sc, qc = [spark.table(t).count() for t in (b, s, q)]
    print(f"{name}: bronze {bc} = silver {sc} + quarantine {qc}")
    add(name, "bronze = silver + quarantine", abs(bc - (sc + qc)))

# 2. keys: every Bronze key is in Silver or quarantine
add("trades", "every bronze trade_id is in silver or quarantine", one("""
    SELECT COUNT(*) FROM (SELECT DISTINCT UPPER(TRIM(trade_id)) AS k FROM bronze.trades) b
    LEFT ANTI JOIN (SELECT trade_id AS k FROM silver.trades
                    UNION SELECT trade_id FROM silver.trades_quarantine) a ON b.k = a.k"""))

add("prices", "every bronze security+date is in silver or quarantine", one("""
    SELECT COUNT(*) FROM (SELECT DISTINCT CONCAT(UPPER(TRIM(security_id)), '|', TRIM(price_date)) AS k FROM bronze.prices) b
    LEFT ANTI JOIN (SELECT CONCAT(security_id, '|', CAST(price_date AS STRING)) AS k FROM silver.prices
                    UNION SELECT CONCAT(security_id, '|', TRIM(price_date)) FROM silver.prices_quarantine) a ON b.k = a.k"""))

# 3. Silver keys are unique
add("trades", "silver trade_id unique", one(
    "SELECT COUNT(*) FROM (SELECT trade_id FROM silver.trades GROUP BY trade_id HAVING COUNT(*) > 1)"))
add("prices", "silver security_id + price_date unique", one(
    "SELECT COUNT(*) FROM (SELECT security_id, price_date FROM silver.prices GROUP BY security_id, price_date HAVING COUNT(*) > 1)"))

# 4. every quarantined row has a reason
add("trades", "quarantine rows have a reject_reason", one(
    "SELECT COUNT(*) FROM silver.trades_quarantine WHERE reject_reason IS NULL"))
add("prices", "quarantine rows have a reject_reason", one(
    "SELECT COUNT(*) FROM silver.prices_quarantine WHERE reject_reason IS NULL"))

df = spark.createDataFrame(results, ["table_name", "rule", "failing_rows", "status", "run_ts"])
display(df)

In [0]:
df.write.mode("append").saveAsTable("dq.results")